In [1]:
from tensorflow.keras.datasets import mnist
import numpy as np
from Seera_init import tensor as Tensor
import matplotlib.pyplot as plt
from Seera import (
    Input, Conv2D, MaxPool2D, Flatten, Dense,
    Sequential, Loss, Adam,
)
from Seera_Engine import autograd4nn


2026-04-10 19:42:24.284268: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-10 19:42:24.318562: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-10 19:42:25.131409: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train_ = np.expand_dims(X_train,axis=1)/255
y_train_ = np.zeros((60000,10))
for i in range (0,60000):
    y_train_[i,:] = np.eye(1,10,y_train[i])

In [3]:
model = Sequential([
    Input((1, 28, 28)),
    Conv2D(8, 1, (3, 3), activation="relu", stride=1, zero_padding=1),
    MaxPool2D(pool_size=(2, 2), stride=2),
    Conv2D(16, 8, (3, 3), activation="relu", stride=1, zero_padding=1),
    MaxPool2D(pool_size=(2, 2), stride=2),
    Flatten(),
    Dense(16*7*7, 128, activation="relu"),
    Dense(128, 16, activation="relu"),
    
    Dense(16, 10, activation="softmax"),
], device="cuda")
model.summary()

Model Summary
  Layer 0: Input Layer with shape (1, 28, 28)
  Layer 1: Conv2D(1→8, kernel=(3, 3), act=relu)
  Layer 2: MaxPool2D(pool=(2, 2), stride=(2, 2))
  Layer 3: Conv2D(8→16, kernel=(3, 3), act=relu)
  Layer 4: MaxPool2D(pool=(2, 2), stride=(2, 2))
  Layer 5: Flatten Layer
  Layer 6: Dense(784→128, act=relu, params=100480)
  Layer 7: Dense(128→16, act=relu, params=2064)
  Layer 8: Dense(16→10, act=softmax, params=170)


In [4]:
loss_fn = Loss()
optimizer = Adam(model, lr=1e-4)

In [5]:
idx = np.random.permutation(len(X_train_))
X, y = X_train_[idx], y_train_[idx]  # shuffle (assuming labels y)

In [6]:

batch_size = 16
epochs = 3  
loss_track = 0.0
for epoch in range(epochs):
    epoch_loss = 0.0
    n_batches = 0
    for i in range(0, 25000, batch_size):
        X_batch, y_batch = X[i:i+batch_size], y[i:i+batch_size]
        X_batch = Tensor(X_batch, is_leaf=True, device="cuda")
        Y_batch = Tensor(y_batch, device="cuda")
        if(i%(1600) ==0 ):
            print(y_batch)
            print("#########down######")
            print(Y_batch.value.to_host_f32())
        pred = model.forward(X_batch)
        loss = loss_fn.categorical_cross_entropy(pred, Y_batch)
        loss_val = float(loss.value.to_host_f32().ravel()[0])
        epoch_loss += loss_val
        n_batches += 1
        del(X_batch)
        del(Y_batch)

        model.zero_grad()
        autograd4nn(loss)
        optimizer.step()
        
        
    avg_loss = epoch_loss / n_batches
    print(f"EPOCH {epoch+1}/{epochs}: Loss: {avg_loss:.10f}")

[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]]
#########down######
[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.

In [8]:
X_test_ = np.expand_dims(X_test,axis=1)/255


In [9]:
correct = 0
for i in range(len(X_test)):
    x = Tensor(X_test_[i:i+1], is_leaf=True, device="cuda")
    pred = model.predict(x)
    # bring to host for argmax
    pred_np = pred.value.to_host_f32()
    pred_label = np.argmax(pred_np)
    if pred_label == y_test[i]:
        correct += 1

accuracy = correct / len(X_test) * 100
print(f"\nTest Accuracy: {accuracy:.1f}% ({correct}/{len(X_test)})")
print("GPU test complete ✓")


Test Accuracy: 96.5% (9649/10000)
GPU test complete ✓
